In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

import optax
from optax.losses import huber_loss
from flax import nnx
import orbax.checkpoint as ocp
import jraph

import diffrax
from diffrax import diffeqsolve, ODETerm, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize

import jaxpm
from jaxpm import camels, plotting, hpm, nn, graph, diagnostics, objectives

print(jax.default_backend())

/global/common/software/des/athomsen/flatiron/lib/python3.11/site-packages/jax_cosmo/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound


gpu


In [6]:
@jax.jit
def solve_ode(y0, t0, ts, dt0=0.01, gravity_model=None, pressure_model=None):
    ode = ODETerm(
        hpm.get_hpm_network_ode_fn(
            mesh_per_dim, 
            cosmo, 
            gravity_model=gravity_model, 
            pressure_model=pressure_model, 
        )
    )

    res = diffeqsolve(
            terms=ode,
            solver=LeapfrogMidpoint(),
            t0=t0,
            t1=ts[-1],
            dt0=dt0,
            y0=y0,
            saveat=SaveAt(ts=ts),
            max_steps=1000,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys

    return res



In [13]:
def get_ode(parts_per_dim, mesh_per_dim):
    train_dict = camels.load_CV_snapshots(
        "CV_0",
        mesh_per_dim,
        parts_per_dim,
        i_snapshots=None,
        CAMELS="/pscratch/sd/a/athomsen/flatiron/CAMELS",
        CODE="SIMBA",
    )
    
    # general
    cosmo = train_dict["cosmo"]
    scales = train_dict["scales"]
    
    # particles
    dm_poss = train_dict["dm_poss"]
    dm_vels = train_dict["dm_vels"]
    
    gas_poss = train_dict["gas_poss"]
    gas_vels = train_dict["gas_vels"]
    
    i0 = 0
    y0 = (dm_poss[i0], dm_vels[i0], gas_poss[i0], gas_vels[i0])
    t0 = scales[i0]
    ts = scales

    return lambda: solve_ode(y0, t0, ts)

In [16]:
ode_64_64 = get_ode(64, 64)
temp = ode_64_64()

Loaded /pscratch/sd/a/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_0/parts=64,mesh=64.h5


In [18]:
%%timeit
ode_64_64()

59 ms ± 25 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [19]:
ode_128_128 = get_ode(128, 128)
temp = ode_128_128()

Loaded /pscratch/sd/a/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_0/parts=128,mesh=128.h5
dark matter and gas
Inferred pressure_model architecture None
dark matter and gas
Inferred pressure_model architecture None
dark matter and gas
Inferred pressure_model architecture None


In [20]:
%%timeit
ode_128_128()

429 ms ± 294 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
ode_256_256 = get_ode(None, 256)
temp = ode_256_256()

Creating /pscratch/sd/a/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_0/parts=None,mesh=256.h5
Found matching catalogs
Using all particles


loading snapshots:  91%|█████████ | 31/34 [06:10<00:43, 14.50s/it]

In [20]:
%%timeit
ode_256_256()

429 ms ± 294 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
